In [1]:
import pystac_client
import planetary_computer as pc
import odc.stac
from shapely.geometry import box, mapping, shape, Point
from IPython.display import Image
import rioxarray
from urllib.parse import urlparse, parse_qs
import xarray as xr 
import numpy as np

In [2]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

In [3]:
reference_point=Point(-63.90, -8.76)
recent_window = "2023-06-01/2023-06-30"
bbox=[-63.9104673, -9.135193, -62.9111701, -8.141095]

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=recent_window,
    max_items=100,
)

items=list(search.items())
best_item = min(items, key=lambda item: item.properties["eo:cloud_cover"])
print(best_item.id)

S2B_MSIL2A_20230629T142719_R053_T20LNQ_20230629T214732


In [4]:
import odc.stac

ds = odc.stac.load(
    [best_item],
    bands=["B04", "B08"],
    bbox=bbox,
    resolution=10,
)
ds

<xarray.Dataset> Size: 969MB
Dimensions:      (y: 11004, x: 11010, time: 1)
Coordinates:
  * y            (y) float64 88kB 9.1e+06 9.1e+06 9.1e+06 ... 8.99e+06 8.99e+06
  * x            (x) float64 88kB 3.997e+05 3.997e+05 ... 5.098e+05 5.098e+05
  * time         (time) datetime64[us] 8B 2023-06-29T14:27:19.024000
    spatial_ref  int32 4B 32720
Data variables:
    B04          (time, y, x) float32 485MB nan nan nan ... 1.301e+03 1.288e+03
    B08          (time, y, x) float32 485MB nan nan nan ... 3.1e+03 3.282e+03

In [5]:
baseline = best_item.properties["s2:processing_baseline"]
offset = -1000 if float(baseline) >= 4.0 else 0   
print("baseline:", baseline, "-> offset:", offset)

red = ds["B04"].isel(time=0)
nir = ds["B08"].isel(time=0)


red = (red.where(red != 0) + offset) / 10000
nir = (nir.where(nir != 0) + offset) / 10000

ndvi = (nir - red) / (nir + red)
ndvi

baseline: 05.09 -> offset: -1000


<xarray.DataArray (y: 11004, x: 11010)> Size: 485MB
array([[       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       ...,
       [       nan,        nan,        nan, ..., 0.82234603, 0.81627977,
        0.81533223],
       [       nan,        nan,        nan, ..., 0.79744345, 0.7502296 ,
        0.77377045],
       [       nan,        nan,        nan, ..., 0.77370834, 0.7492711 ,
        0.7758755 ]], shape=(11004, 11010), dtype=float32)
Coordinates:
  * y            (y) float64 88kB 9.1e+06 9.1e+06 9.1e+06 ... 8.99e+06 8.99e+06
  * x            (x) float64 88kB 3.997e+05 3.997e+05 ... 5.098e+05 5.098e+05
    spatial_ref  int32 4B 32720
    time         datetime64[us] 8B 2023-06-29T14:27:19.024000

In [7]:
pixel= ndvi.isel(y=0,x=0)
pixel

<xarray.DataArray ()> Size: 4B
array(nan, dtype=float32)
Coordinates:
    y            float64 8B 9.1e+06
    x            float64 8B 3.997e+05
    spatial_ref  int32 4B 32720
    time         datetime64[us] 8B 2023-06-29T14:27:19.024000

In [8]:
window= ndvi.isel(y=slice(0, 100), x=slice(0, 100))
window.shape

(100, 100)

In [9]:
x_centro = float(ndvi.x.mean())
y_centro = float(ndvi.y.mean())

ndvi.sel(x=x_centro, y=y_centro, method="nearest")

<xarray.DataArray ()> Size: 4B
array(nan, dtype=float32)
Coordinates:
    y            float64 8B 9.045e+06
    x            float64 8B 4.547e+05
    spatial_ref  int32 4B 32720
    time         datetime64[us] 8B 2023-06-29T14:27:19.024000

In [17]:
crs=ndvi.rio.crs
crs

CRS.from_wkt('PROJCS["WGS 84 / UTM zone 20S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-63],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32720"]]')

In [10]:
res=ndvi.rio.resolution()
res

(10.0, -10.0)

In [11]:
trans=ndvi.rio.transform()
trans

Affine(10.0, 0.0, 399690.0,
       0.0, -10.0, 9100110.0)

In [12]:
ndvi_4326 = ndvi.rio.reproject("EPSG:4326")

In [13]:
from pathlib import Path

out_dir = Path("data")                 # cartella accanto al notebook
out_dir.mkdir(exist_ok=True)           # la crea se non esiste

ndvi.astype("float32").rio.to_raster(
    out_dir / "ndvi_day2.tif",
    compress="deflate",                # file più piccolo, senza perdita
)